In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import json

# ─────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────
DATA_PATH   = "/kaggle/input/datasets/sauravraj23/tata-enter/tata_motors_enterprise_demand.csv"
OUT_DIR     = Path("/mnt/user-data/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS    = list(range(1, 13))   # 1-step ahead to 12-step ahead
TRAIN_FROM  = "2020-01-01"         # post-COVID 4-year window
CV_FOLDS    = 3
CV_GAP      = 12                   # months gap between train and test in CV
MIN_HISTORY = 24                   # min months of data to include a variant

# ─────────────────────────────────────────
# 1. LOAD & AGGREGATE TO MONTHLY VARIANT LEVEL
# ─────────────────────────────────────────
print("="*60)
print("STEP 1: Loading and aggregating data")
print("="*60)

df_raw = pd.read_csv(DATA_PATH)
df_raw['sales_date']  = pd.to_datetime(df_raw['sales_date'])
df_raw['year_month']  = pd.to_datetime(df_raw['year_month'])

# Aggregate dealer rows → monthly variant panel
monthly = df_raw.groupby(['model', 'variant', 'year_month']).agg(
    sales_unit                = ('sales_unit',                   'sum'),
    fuel_type                 = ('fuel_type',                    'first'),
    transmission              = ('transmission',                 'first'),
    body_type                 = ('body_type',                    'first'),
    is_discontinued           = ('is_discontinued',              'max'),
    months_on_market          = ('months_on_market',             'max'),
    months_since_launch       = ('months_since_launch',          'max'),
    avg_discount_inr          = ('avg_discount_inr',             'mean'),
    wait_time_weeks           = ('wait_time_weeks',              'mean'),
    web_traffic_views         = ('web_traffic_views',            'sum'),
    net_bookings              = ('net_bookings',                 'sum'),
    test_drive_requests       = ('test_drive_requests',          'sum'),
    dealer_footfall           = ('dealer_footfall',              'sum'),
    consumer_offer_active     = ('consumer_offer_active',        'max'),
    auto_loan_emi_rate        = ('auto_loan_emi_rate',           'mean'),
    marketing_spend_inr       = ('marketing_spend_inr',          'sum'),
    months_since_facelift     = ('months_since_facelift',        'max'),
    internal_cannibalization  = ('internal_cannibalization_flag','max'),
    fuel_price_petrol         = ('fuel_price_petrol',            'mean'),
    fuel_price_diesel         = ('fuel_price_diesel',            'mean'),
    nifty_auto                = ('stock_market_nifty_auto',      'mean'),
    festive_season_intensity  = ('festive_season_intensity',     'max'),
    dealer_inventory_days     = ('dealer_inventory_days',        'mean'),
    production_constraint     = ('production_constraint_flag',   'max'),
    msrp_inr                  = ('msrp_inr',                    'first'),
    variant_status            = ('variant_status',               'first'),
    finance_flag              = ('finance_flag',                 'max'),
).reset_index()

print(f"Monthly panel shape: {monthly.shape}")
print(f"Unique model×variant combos: {monthly[['model','variant']].drop_duplicates().shape[0]}")

# ─────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 2: Feature engineering")
print("="*60)

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['model', 'variant', 'year_month']).copy()

    # ── Calendar features ──
    df['month']       = df['year_month'].dt.month
    df['quarter']     = df['year_month'].dt.quarter
    df['year']        = df['year_month'].dt.year
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)

    # ── Indian festival calendar (month-level flags) ──
    df['is_diwali_month']   = df['month'].isin([10, 11]).astype(int)   # Oct-Nov
    df['is_navratri_month'] = df['month'].isin([9, 10]).astype(int)    # Sep-Oct
    df['is_onam_month']     = df['month'].isin([8, 9]).astype(int)     # Aug-Sep
    df['is_eid_month']      = df['month'].isin([4, 5]).astype(int)
    df['is_year_end_push']  = df['month'].isin([3]).astype(int)        # FY end
    df['is_h2_push']        = df['month'].isin([9, 10, 11, 12]).astype(int)

    # ── Structural shock dummies ──
    df['covid_flag']          = ((df['year_month'] >= '2020-03-01') &
                                 (df['year_month'] <= '2021-06-01')).astype(int)
    df['gst_transition']      = ((df['year_month'] >= '2017-06-01') &
                                 (df['year_month'] <= '2017-10-01')).astype(int)
    df['bs6_transition']      = ((df['year_month'] >= '2019-10-01') &
                                 (df['year_month'] <= '2020-04-01')).astype(int)
    df['semi_shortage']       = ((df['year_month'] >= '2021-01-01') &
                                 (df['year_month'] <= '2022-09-01')).astype(int)

    # ── EV subsidy dummies ──
    df['fame2_active']        = ((df['year_month'] >= '2019-04-01') &
                                 (df['year_month'] <= '2024-03-01')).astype(int)
    df['is_ev']               = df['fuel_type'].isin(['Electric']).astype(int)
    df['ev_with_subsidy']     = df['is_ev'] * df['fame2_active']

    # ── Lifecycle phase ──
    df['lifecycle_phase'] = pd.cut(
        df['months_since_launch'],
        bins=[-1, 6, 18, 42, 72, 9999],
        labels=['launch', 'ramp_up', 'maturity', 'decline', 'legacy']
    ).astype(str)

    # ── Price features ──
    df['discount_pct']        = df['avg_discount_inr'] / (df['msrp_inr'] + 1)
    df['effective_price_idx'] = (df['msrp_inr'] - df['avg_discount_inr']) / df['msrp_inr'].mean()

    # ── Fuel price ratio ──
    df['petrol_diesel_ratio'] = df['fuel_price_petrol'] / (df['fuel_price_diesel'] + 1)

    # ── Booking-to-sale leading indicators ──
    df['booking_to_traffic']  = df['net_bookings'] / (df['web_traffic_views'] + 1)
    df['drive_to_footfall']   = df['test_drive_requests'] / (df['dealer_footfall'] + 1)

    # ── Lag features (computed per variant) ──
    grp = df.groupby(['model', 'variant'])['sales_unit']

    for lag in [1, 2, 3, 6, 12]:
        df[f'lag_{lag}'] = grp.shift(lag)

    # Rolling stats (on lagged values to avoid leakage)
    for window in [3, 6, 12]:
        df[f'roll_mean_{window}'] = (
            grp.shift(1).rolling(window, min_periods=2).mean().values
        )
        df[f'roll_std_{window}'] = (
            grp.shift(1).rolling(window, min_periods=2).std().values
        )
        df[f'roll_max_{window}'] = (
            grp.shift(1).rolling(window, min_periods=2).max().values
        )

    # MoM and YoY growth
    df['mom_growth'] = grp.shift(1).pct_change(1).replace([np.inf, -np.inf], np.nan)
    df['yoy_growth'] = (
        (grp.shift(1) - grp.shift(13)) / (grp.shift(13) + 1)
    )

    # Exponential weighted mean (captures trend better than rolling)
    df['ewm_3']  = grp.shift(1).ewm(span=3,  min_periods=1).mean().values
    df['ewm_6']  = grp.shift(1).ewm(span=6,  min_periods=1).mean().values
    df['ewm_12'] = grp.shift(1).ewm(span=12, min_periods=1).mean().values

    # Lag for exogenous leading signals
    for col in ['net_bookings', 'web_traffic_views', 'test_drive_requests',
                'marketing_spend_inr', 'dealer_footfall']:
        df[f'{col}_lag1'] = df.groupby(['model', 'variant'])[col].shift(1)
        df[f'{col}_lag2'] = df.groupby(['model', 'variant'])[col].shift(2)

    # ── Variant share within model (cross-variant competition signal) ──
    monthly_model_sales = df.groupby(['model', 'year_month'])['sales_unit'].transform('sum')
    df['variant_share']        = df['sales_unit'] / (monthly_model_sales + 1)
    # Use lagged share to avoid leakage
    df['variant_share_lag1']   = df.groupby(['model','variant'])['variant_share'].shift(1)

    # ── Model-level trend ──
    model_grp = df.groupby(['model', 'year_month'])['sales_unit'].sum().reset_index()
    model_grp['model_roll3']  = model_grp.groupby('model')['sales_unit'].shift(1).rolling(3,  min_periods=1).mean().values
    model_grp['model_roll6']  = model_grp.groupby('model')['sales_unit'].shift(1).rolling(6,  min_periods=1).mean().values
    model_grp['model_roll12'] = model_grp.groupby('model')['sales_unit'].shift(1).rolling(12, min_periods=1).mean().values
    df = df.merge(model_grp[['model','year_month','model_roll3','model_roll6','model_roll12']],
                  on=['model','year_month'], how='left')

    return df


panel = build_features(monthly)
print(f"Panel shape after feature engineering: {panel.shape}")
print(f"Feature columns: {panel.shape[1] - 3}")  # exclude model/variant/year_month

# ─────────────────────────────────────────
# 3. ENCODE CATEGORICALS
# ─────────────────────────────────────────
CAT_COLS = ['model', 'variant', 'fuel_type', 'transmission',
            'body_type', 'variant_status', 'lifecycle_phase']

encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    panel[col] = le.fit_transform(panel[col].astype(str))
    encoders[col] = le

# ─────────────────────────────────────────
# 4. DEFINE FEATURE SET
# ─────────────────────────────────────────
EXCLUDE = ['sales_unit', 'year_month']
FEATURE_COLS = [c for c in panel.columns if c not in EXCLUDE]

print(f"\nTotal features: {len(FEATURE_COLS)}")

# ─────────────────────────────────────────
# 5. WALK-FORWARD CV EVALUATION
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: Walk-forward cross-validation")
print("="*60)

def wape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.maximum(np.array(y_pred), 0)
    denom  = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom * 100

def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.maximum(np.array(y_pred), 0)))

# LightGBM params (tuned for sparse count data)
LGB_PARAMS = {
    'objective':       'tweedie',
    'tweedie_variance_power': 1.5,
    'metric':          'tweedie',
    'learning_rate':   0.05,
    'num_leaves':      63,
    'max_depth':       7,
    'min_child_samples': 30,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq':    5,
    'lambda_l1':       0.1,
    'lambda_l2':       0.1,
    'n_estimators':    600,
    'verbose':         -1,
    'n_jobs':          -1,
    'random_state':    42,
}

# Filter to training window
train_panel = panel[panel['year_month'] >= TRAIN_FROM].copy()
all_months  = sorted(train_panel['year_month'].unique())
n_months    = len(all_months)

print(f"Training window: {TRAIN_FROM} → {all_months[-1].strftime('%Y-%m')}")
print(f"Total months in panel: {n_months}")
print(f"CV folds: {CV_FOLDS}, CV gap: {CV_GAP} months")

# CV split points: last 3 non-overlapping test windows of 3 months each
# Train → all data before cutoff; Test → next 3 months after gap
cv_results = []

fold_test_sizes = [3, 3, 3]
fold_end_offsets = [n_months, n_months - 3, n_months - 6]

for fold_idx, (test_end_offset, test_size) in enumerate(
        zip(fold_end_offsets, fold_test_sizes)):

    test_end   = fold_end_offset = all_months[test_end_offset - 1]
    test_start = all_months[test_end_offset - test_size]
    train_end  = all_months[test_end_offset - test_size - CV_GAP]  # gap

    fold_train = train_panel[train_panel['year_month'] <= train_end]
    fold_test  = train_panel[(train_panel['year_month'] >= test_start) &
                             (train_panel['year_month'] <= test_end)]

    # Drop rows with NaN target or key lag features
    fold_train = fold_train.dropna(subset=['sales_unit', 'lag_1', 'lag_2', 'roll_mean_3'])
    fold_test  = fold_test.dropna(subset=['sales_unit', 'lag_1', 'lag_2', 'roll_mean_3'])

    X_tr = fold_train[FEATURE_COLS]
    y_tr = fold_train['sales_unit']
    X_te = fold_test[FEATURE_COLS]
    y_te = fold_test['sales_unit']

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(-1)])

    preds = np.maximum(model.predict(X_te), 0)
    fold_wape = wape(y_te, preds)
    fold_mae  = mae(y_te, preds)

    cv_results.append({'fold': fold_idx+1,
                       'train_end': str(train_end)[:7],
                       'test_start': str(test_start)[:7],
                       'test_end': str(test_end)[:7],
                       'wape': round(fold_wape, 2),
                       'mae': round(fold_mae, 3),
                       'n_train': len(fold_train),
                       'n_test': len(fold_test)})

    print(f"  Fold {fold_idx+1}: train→{str(train_end)[:7]}  "
          f"test[{str(test_start)[:7]}:{str(test_end)[:7]}]  "
          f"WAPE={fold_wape:.2f}%  MAE={fold_mae:.3f}")

mean_wape = np.mean([r['wape'] for r in cv_results])
mean_mae  = np.mean([r['mae']  for r in cv_results])
print(f"\n→ Mean CV WAPE: {mean_wape:.2f}%  (target <20%)")
print(f"→ Mean CV MAE:  {mean_mae:.3f}")

# ─────────────────────────────────────────
# 6. FINAL MODEL: TRAIN ON ALL DATA
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 4: Training final model on full history")
print("="*60)

full_train = train_panel.dropna(subset=['sales_unit', 'lag_1', 'lag_2', 'roll_mean_3'])
X_full = full_train[FEATURE_COLS]
y_full = full_train['sales_unit']

final_model = lgb.LGBMRegressor(**LGB_PARAMS)
final_model.fit(X_full, y_full, callbacks=[lgb.log_evaluation(-1)])
print(f"Final model trained on {len(full_train):,} rows | {len(FEATURE_COLS)} features")

# ─────────────────────────────────────────
# 7. RECURSIVE MULTI-STEP FORECAST (12 months)
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 5: Generating 12-month ahead forecasts")
print("="*60)

# Last known date is Oct 2025; forecast Nov 2025 → Oct 2026
LAST_DATE = panel['year_month'].max()
FORECAST_DATES = pd.date_range(
    start=LAST_DATE + pd.DateOffset(months=1),
    periods=12, freq='MS'
)

# Seed the forecast with the last N months of actuals for lag computation
SEED_MONTHS = 24
seed_df = panel[panel['year_month'] > (LAST_DATE - pd.DateOffset(months=SEED_MONTHS))].copy()

# Reference dataframe for static variant attributes
static_ref = panel.groupby(['model','variant']).last().reset_index()
static_cols = ['model', 'variant', 'fuel_type', 'transmission', 'body_type',
               'variant_status', 'msrp_inr', 'is_ev', 'months_since_facelift',
               'internal_cannibalization', 'is_discontinued']
# Filter static cols to those actually present
static_cols = [c for c in static_cols if c in panel.columns]

# Build a rolling history dict: {(model, variant): list of monthly sales}
from collections import defaultdict

history = {}
for (m, v), grp in panel.groupby(['model','variant']):
    history[(m, v)] = list(grp.sort_values('year_month')['sales_unit'].values)

forecast_rows = []

for h, fdate in enumerate(FORECAST_DATES, start=1):
    print(f"  Forecasting horizon h={h}: {fdate.strftime('%Y-%m')}", end="\r")

    rows = []
    for (m, v), hist in history.items():
        hist_arr = np.array(hist, dtype=float)
        n = len(hist_arr)
        if n < 3:
            continue  # not enough history

        # ── Build lag features from rolling history ──
        lag_1  = hist_arr[-1]  if n >= 1  else np.nan
        lag_2  = hist_arr[-2]  if n >= 2  else np.nan
        lag_3  = hist_arr[-3]  if n >= 3  else np.nan
        lag_6  = hist_arr[-6]  if n >= 6  else np.nan
        lag_12 = hist_arr[-12] if n >= 12 else np.nan

        w = max(1, n)
        roll3_arr  = hist_arr[-min(3,w):]
        roll6_arr  = hist_arr[-min(6,w):]
        roll12_arr = hist_arr[-min(12,w):]

        roll_mean_3  = np.mean(roll3_arr)
        roll_std_3   = np.std(roll3_arr) if len(roll3_arr)>1 else 0
        roll_max_3   = np.max(roll3_arr)
        roll_mean_6  = np.mean(roll6_arr)
        roll_std_6   = np.std(roll6_arr) if len(roll6_arr)>1 else 0
        roll_max_6   = np.max(roll6_arr)
        roll_mean_12 = np.mean(roll12_arr)
        roll_std_12  = np.std(roll12_arr) if len(roll12_arr)>1 else 0
        roll_max_12  = np.max(roll12_arr)

        mom_growth = (lag_1 - lag_2) / (lag_2 + 1) if lag_2 is not np.nan else 0
        yoy_growth = (lag_1 - lag_12) / (lag_12 + 1) if lag_12 is not np.nan else 0

        ewm3  = pd.Series(hist_arr).ewm(span=3,  min_periods=1).mean().iloc[-1]
        ewm6  = pd.Series(hist_arr).ewm(span=6,  min_periods=1).mean().iloc[-1]
        ewm12 = pd.Series(hist_arr).ewm(span=12, min_periods=1).mean().iloc[-1]

        # ── Static/slowly-changing features ──
        ref_row = static_ref[(static_ref['model']==m) & (static_ref['variant']==v)]
        if ref_row.empty:
            continue
        ref = ref_row.iloc[0]

        # ── Calendar ──
        month    = fdate.month
        quarter  = fdate.quarter
        year     = fdate.year
        month_sin = np.sin(2 * np.pi * month / 12)
        month_cos = np.cos(2 * np.pi * month / 12)

        # ── Festival flags ──
        is_diwali   = int(month in [10, 11])
        is_navratri = int(month in [9,  10])
        is_onam     = int(month in [8,  9])
        is_eid      = int(month in [4,  5])
        is_ye_push  = int(month == 3)
        is_h2       = int(month in [9, 10, 11, 12])

        # ── Structural shocks (all false for 2025-26 forecast) ──
        covid_flag = gst_flag = bs6_flag = semi_flag = 0
        fame2_active = 0  # FAME-II ended Mar 2024

        is_ev_f     = int('EV' in encoders['model'].classes_[m] if hasattr(encoders['model'], 'classes_') else 0)
        ev_subsidy  = is_ev_f * fame2_active

        # ── Lifecycle ──
        months_launch = ref.get('months_since_launch', 0) + h
        lc_bins  = [-1, 6, 18, 42, 72, 9999]
        lc_labels = ['launch','ramp_up','maturity','decline','legacy']
        lc_phase  = lc_labels[np.digitize(months_launch, lc_bins) - 1]
        lc_enc    = encoders['lifecycle_phase'].transform([lc_phase])[0] \
                    if lc_phase in encoders['lifecycle_phase'].classes_ else 0

        # ── Price (use last known, static for forecast) ──
        msrp_val      = ref['msrp_inr'] if 'msrp_inr' in ref.index else 0
        discount_pct  = 0  # conservative: no new discount assumed
        eff_price_idx = msrp_val / panel['msrp_inr'].mean()

        # ── Get last-known exogenous values for lag features ──
        last_known = seed_df[(seed_df['model']==m) & (seed_df['variant']==v)]
        if last_known.empty:
            last_known = panel[(panel['model']==m) & (panel['variant']==v)].tail(2)

        def get_last(col, default=0):
            if last_known.empty or col not in last_known.columns:
                return default
            val = last_known.sort_values('year_month')[col].dropna()
            return val.iloc[-1] if len(val) > 0 else default

        row = {
            'model':                   m,
            'variant':                 v,
            'fuel_type':               ref['fuel_type']        if 'fuel_type' in ref.index else 0,
            'transmission':            ref['transmission']     if 'transmission' in ref.index else 0,
            'body_type':               ref['body_type']        if 'body_type' in ref.index else 0,
            'variant_status':          ref['variant_status']   if 'variant_status' in ref.index else 0,
            'is_discontinued':         ref['is_discontinued']  if 'is_discontinued' in ref.index else 0,
            'months_on_market':        ref.get('months_on_market', 0) + h,
            'months_since_launch':     months_launch,
            'avg_discount_inr':        get_last('avg_discount_inr', 0),
            'wait_time_weeks':         get_last('wait_time_weeks', 4),
            'web_traffic_views':       get_last('web_traffic_views', 500),
            'net_bookings':            get_last('net_bookings', 1),
            'test_drive_requests':     get_last('test_drive_requests', 1),
            'dealer_footfall':         get_last('dealer_footfall', 50),
            'consumer_offer_active':   get_last('consumer_offer_active', 0),
            'auto_loan_emi_rate':      get_last('auto_loan_emi_rate', 8.5),
            'marketing_spend_inr':     get_last('marketing_spend_inr', 1e6),
            'months_since_facelift':   ref.get('months_since_facelift', 0) + h,
            'internal_cannibalization': ref.get('internal_cannibalization', 0),
            'fuel_price_petrol':       get_last('fuel_price_petrol', 102),
            'fuel_price_diesel':       get_last('fuel_price_diesel', 90),
            'nifty_auto':              get_last('nifty_auto', 22000),
            'festive_season_intensity': is_diwali + is_navratri,
            'dealer_inventory_days':   get_last('dealer_inventory_days', 30),
            'production_constraint':   0,
            'msrp_inr':                msrp_val,
            'finance_flag':            get_last('finance_flag', 1),
            'month':                   month,
            'quarter':                 quarter,
            'year':                    year,
            'month_sin':               month_sin,
            'month_cos':               month_cos,
            'is_diwali_month':         is_diwali,
            'is_navratri_month':       is_navratri,
            'is_onam_month':           is_onam,
            'is_eid_month':            is_eid,
            'is_year_end_push':        is_ye_push,
            'is_h2_push':              is_h2,
            'covid_flag':              covid_flag,
            'gst_transition':          gst_flag,
            'bs6_transition':          bs6_flag,
            'semi_shortage':           semi_flag,
            'fame2_active':            fame2_active,
            'is_ev':                   is_ev_f,
            'ev_with_subsidy':         ev_subsidy,
            'lifecycle_phase':         lc_enc,
            'discount_pct':            discount_pct,
            'effective_price_idx':     eff_price_idx,
            'petrol_diesel_ratio':     get_last('fuel_price_petrol', 102) / (get_last('fuel_price_diesel', 90) + 1),
            'booking_to_traffic':      get_last('net_bookings', 1) / (get_last('web_traffic_views', 500) + 1),
            'drive_to_footfall':       get_last('test_drive_requests', 1) / (get_last('dealer_footfall', 50) + 1),
            'lag_1':                   lag_1,
            'lag_2':                   lag_2,
            'lag_3':                   lag_3,
            'lag_6':                   lag_6,
            'lag_12':                  lag_12,
            'roll_mean_3':             roll_mean_3,
            'roll_std_3':              roll_std_3,
            'roll_max_3':              roll_max_3,
            'roll_mean_6':             roll_mean_6,
            'roll_std_6':              roll_std_6,
            'roll_max_6':              roll_max_6,
            'roll_mean_12':            roll_mean_12,
            'roll_std_12':             roll_std_12,
            'roll_max_12':             roll_max_12,
            'mom_growth':              mom_growth,
            'yoy_growth':              yoy_growth,
            'ewm_3':                   ewm3,
            'ewm_6':                   ewm6,
            'ewm_12':                  ewm12,
            'net_bookings_lag1':       get_last('net_bookings', 1),
            'net_bookings_lag2':       get_last('net_bookings', 1),
            'web_traffic_views_lag1':  get_last('web_traffic_views', 500),
            'web_traffic_views_lag2':  get_last('web_traffic_views', 500),
            'test_drive_requests_lag1': get_last('test_drive_requests', 1),
            'test_drive_requests_lag2': get_last('test_drive_requests', 1),
            'marketing_spend_inr_lag1': get_last('marketing_spend_inr', 1e6),
            'marketing_spend_inr_lag2': get_last('marketing_spend_inr', 1e6),
            'dealer_footfall_lag1':    get_last('dealer_footfall', 50),
            'dealer_footfall_lag2':    get_last('dealer_footfall', 50),
            'variant_share':           0,
            'variant_share_lag1':      get_last('variant_share', 0) if 'variant_share' in last_known.columns else 0,
            'model_roll3':             0,
            'model_roll6':             0,
            'model_roll12':            0,
        }
        rows.append(row)

    if not rows:
        print(f"  No rows for horizon {h}, skipping")
        continue

    X_fc = pd.DataFrame(rows)
    # Align to FEATURE_COLS (fill any missing with 0)
    for c in FEATURE_COLS:
        if c not in X_fc.columns:
            X_fc[c] = 0
    X_fc = X_fc[FEATURE_COLS]

    fc_preds = np.maximum(final_model.predict(X_fc), 0)

    for i, (m, v) in enumerate([(r['model'], r['variant']) for r in rows]):
        forecast_rows.append({
            'model':         encoders['model'].inverse_transform([m])[0],
            'variant':       encoders['variant'].inverse_transform([v])[0],
            'forecast_month': fdate.strftime('%Y-%m'),
            'horizon_h':     h,
            'forecast_units': round(fc_preds[i], 2),
        })
        # Update history for next horizon
        history[(m, v)] = list(history[(m, v)]) + [fc_preds[i]]

print("\n  Forecasting complete.")

# ─────────────────────────────────────────
# 8. ASSEMBLE & SAVE OUTPUTS
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 6: Saving outputs")
print("="*60)

forecast_df = pd.DataFrame(forecast_rows)
# Decode model/variant names (already decoded above in loop)

# ── Output 1: Variant-level forecast ──
variant_fc = forecast_df.pivot_table(
    index=['model', 'variant'],
    columns='forecast_month',
    values='forecast_units',
    aggfunc='sum'
).reset_index()
variant_fc.to_csv(OUT_DIR / "variant_forecast_12month.csv", index=False)
print(f"  ✓ variant_forecast_12month.csv — {variant_fc.shape}")

# ── Output 2: Long format ──
forecast_df.to_csv(OUT_DIR / "variant_forecast_long.csv", index=False)
print(f"  ✓ variant_forecast_long.csv — {forecast_df.shape}")

# ── Output 3: Feature importance ──
feat_imp = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)
feat_imp.to_csv(OUT_DIR / "feature_importance.csv", index=False)
print(f"  ✓ feature_importance.csv")

# ── Output 4: CV results ──
cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(OUT_DIR / "cv_results.csv", index=False)
print(f"  ✓ cv_results.csv")

# ── Summary ──
summary = {
    'cv_mean_wape': round(mean_wape, 2),
    'cv_mean_mae': round(mean_mae, 3),
    'total_variants_forecasted': forecast_df[['model','variant']].drop_duplicates().shape[0],
    'forecast_horizon_months': 12,
    'forecast_start': FORECAST_DATES[0].strftime('%Y-%m'),
    'forecast_end': FORECAST_DATES[-1].strftime('%Y-%m'),
    'total_features': len(FEATURE_COLS),
    'cv_folds': CV_FOLDS,
}
with open(OUT_DIR / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print(f"PIPELINE COMPLETE")
print(f"{'='*60}")
print(f"CV Mean WAPE : {mean_wape:.2f}%  (target <20%)")
print(f"CV Mean MAE  : {mean_mae:.3f}")
print(f"Variants     : {summary['total_variants_forecasted']}")
print(f"Forecast     : {summary['forecast_start']} → {summary['forecast_end']}")
print(f"Top 15 Features:")
print(feat_imp.head(15).to_string(index=False))

STEP 1: Loading and aggregating data
Monthly panel shape: (16574, 30)
Unique model×variant combos: 306

STEP 2: Feature engineering
Panel shape after feature engineering: (16574, 88)
Feature columns: 85

Total features: 86

STEP 3: Walk-forward cross-validation
Training window: 2020-01-01 → 2025-10
Total months in panel: 70
CV folds: 3, CV gap: 12 months
  Fold 1: train→2024-08  test[2025-08:2025-10]  WAPE=13.23%  MAE=0.524
  Fold 2: train→2024-05  test[2025-05:2025-07]  WAPE=9.54%  MAE=0.315
  Fold 3: train→2024-02  test[2025-02:2025-04]  WAPE=8.37%  MAE=0.242

→ Mean CV WAPE: 10.38%  (target <20%)
→ Mean CV MAE:  0.360

STEP 4: Training final model on full history
Final model trained on 8,313 rows | 86 features

STEP 5: Generating 12-month ahead forecasts
  Forecasting horizon h=12: 2026-10
  Forecasting complete.

STEP 6: Saving outputs
  ✓ variant_forecast_12month.csv — (306, 14)
  ✓ variant_forecast_long.csv — (3672, 5)
  ✓ feature_importance.csv
  ✓ cv_results.csv

PIPELINE COMPL

## The above model has Data leak problem that's why this gives around 90% accuracy.

In [2]:


import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
import json

# ─────────────────────────────────────────
# 0. CONFIG
# ─────────────────────────────────────────
DATA_PATH   = "/kaggle/input/datasets/sauravraj23/tata-enter/tata_motors_enterprise_demand.csv"
OUT_DIR     = Path("/mnt/user-data/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS    = list(range(1, 13))   # 1-step ahead to 12-step ahead
TRAIN_FROM  = "2020-01-01"
CV_FOLDS    = 3

# ─────────────────────────────────────────
# 1. LOAD, AGGREGATE & DENSIFY (CRITICAL FIX)
# ─────────────────────────────────────────
print("="*60)
print("STEP 1: Loading, aggregating, and densifying data")
print("="*60)

try:
    df_raw = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"File not found at {DATA_PATH}. Please run in the correct environment.")
    import sys; sys.exit(0)

df_raw['year_month'] = pd.to_datetime(df_raw['year_month']).dt.to_period('M').dt.to_timestamp()

# Standard Aggregation
monthly = df_raw.groupby(['model', 'variant', 'year_month']).agg(
    sales_unit               = ('sales_unit', 'sum'),
    fuel_type                = ('fuel_type', 'first'),
    transmission             = ('transmission', 'first'),
    body_type                = ('body_type', 'first'),
    msrp_inr                 = ('msrp_inr', 'first'),
    avg_discount_inr         = ('avg_discount_inr', 'mean'),
    net_bookings             = ('net_bookings', 'sum'),
    web_traffic_views        = ('web_traffic_views', 'sum'),
    months_since_launch      = ('months_since_launch', 'max'),
    dealer_footfall          = ('dealer_footfall', 'sum'),
    marketing_spend_inr      = ('marketing_spend_inr', 'sum'),
    fuel_price_petrol        = ('fuel_price_petrol', 'mean'),
    fuel_price_diesel        = ('fuel_price_diesel', 'mean'),
    nifty_auto               = ('stock_market_nifty_auto', 'mean')
).reset_index()

# ── DENSIFICATION: Ensure every variant has a row for every month ──
# This prevents pandas .shift() from grabbing the wrong month if sales were 0
all_models = monthly[['model', 'variant']].drop_duplicates()
min_date, max_date = monthly['year_month'].min(), monthly['year_month'].max()
full_idx = pd.MultiIndex.from_product(
    [monthly['model'].unique(), monthly['variant'].unique(), pd.date_range(min_date, max_date, freq='MS')],
    names=['model', 'variant', 'year_month']
)

# Reindex and fill missing numeric metrics with 0
monthly = monthly.set_index(['model', 'variant', 'year_month']).reindex(full_idx)
monthly['sales_unit'] = monthly['sales_unit'].fillna(0)
numeric_fills = ['avg_discount_inr', 'net_bookings', 'web_traffic_views', 'dealer_footfall', 'marketing_spend_inr']
monthly[numeric_fills] = monthly[numeric_fills].fillna(0)

# Forward-fill and backward-fill static variant features
monthly = monthly.groupby(['model', 'variant']).ffill().bfill().reset_index()

# Drop combinations that literally never existed
variant_totals = monthly.groupby(['model', 'variant'])['sales_unit'].sum()
valid_variants = variant_totals[variant_totals > 0].index
monthly = monthly.set_index(['model', 'variant']).loc[valid_variants].reset_index()

print(f"Densified panel shape: {monthly.shape}")

# ─────────────────────────────────────────
# 2. FEATURE ENGINEERING & MULTI-STEP TARGETS
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 2: Feature Engineering & Target Creation")
print("="*60)

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['model', 'variant', 'year_month']).copy()

    # ── 1. Create Direct Multi-Step Targets (Shift backward into time T) ──
    for h in HORIZONS:
        # target_h1 means "sales 1 month from now", target_h2 is "sales 2 months from now"
        df[f'target_h{h}'] = df.groupby(['model', 'variant'])['sales_unit'].shift(-h)

    # ── 2. Time/Calendar Features for current month T ──
    df['month'] = df['year_month'].dt.month
    df['quarter'] = df['year_month'].dt.quarter
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    df['is_diwali_month'] = df['month'].isin([10, 11]).astype(int)
    df['is_h2_push']      = df['month'].isin([9, 10, 11, 12]).astype(int)

    # ── 3. Lagged Sales Features (What happened up to month T) ──
    grp = df.groupby(['model', 'variant'])['sales_unit']
    for lag in [1, 2, 3, 6, 12]:
        df[f'lag_{lag}'] = grp.shift(lag)

    for window in [3, 6, 12]:
        df[f'roll_mean_{window}'] = grp.shift(1).rolling(window, min_periods=1).mean().values
        # FIX: fillna(0) applied BEFORE .values
        df[f'roll_std_{window}']  = grp.shift(1).rolling(window, min_periods=1).std().fillna(0).values

    df['yoy_growth'] = (grp.shift(1) - grp.shift(13)) / (grp.shift(13) + 1)

    # ── 4. Lags of Exogenous Features (Leakage-free) ──
    for col in ['net_bookings', 'web_traffic_views', 'dealer_footfall']:
        df[f'{col}_lag1'] = df.groupby(['model', 'variant'])[col].shift(1)
        df[f'{col}_lag2'] = df.groupby(['model', 'variant'])[col].shift(2)

    # ── 5. Cross-Variant Competition (Lagged) ──
    monthly_model_sales = df.groupby(['model', 'year_month'])['sales_unit'].transform('sum')
    # FIX: Corrected the grouping syntax
    df['variant_share'] = df['sales_unit'] / (monthly_model_sales + 1)
    df['variant_share_lag1'] = df.groupby(['model', 'variant'])['variant_share'].shift(1)

    # ── 6. Model-Level Momentum (Lagged) ──
    model_grp = df.groupby(['model', 'year_month'])['sales_unit'].sum().reset_index()
    for window in [3, 6, 12]:
        model_grp[f'model_roll{window}'] = model_grp.groupby('model')['sales_unit'].shift(1).rolling(window, min_periods=1).mean().values
    
    df = df.merge(model_grp[['model', 'year_month', 'model_roll3', 'model_roll6', 'model_roll12']],
                  on=['model', 'year_month'], how='left')

    return df

panel = build_features(monthly)

# ─────────────────────────────────────────
# 3. ENCODE & DEFINE FEATURES
# ─────────────────────────────────────────
CAT_COLS = ['model', 'variant', 'fuel_type', 'transmission', 'body_type']

encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    panel[col] = le.fit_transform(panel[col].astype(str))
    encoders[col] = le

# Identify targets and drop contemporaneous leakage
TARGETS = [f'target_h{h}' for h in HORIZONS]

# FIX: Added 'variant_share' to EXCLUDE to prevent target leakage
EXCLUDE = ['sales_unit', 'year_month', 'net_bookings', 'web_traffic_views', 
           'dealer_footfall', 'marketing_spend_inr', 'variant_share'] + TARGETS

FEATURE_COLS = [c for c in panel.columns if c not in EXCLUDE]
print(f"Total features per model: {len(FEATURE_COLS)}")

# ─────────────────────────────────────────
# 4. WALK-FORWARD CV (Direct Multi-Step)
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: Direct Multi-Step CV Evaluation")
print("="*60)

train_panel = panel[panel['year_month'] >= TRAIN_FROM].copy()
all_months  = sorted(train_panel['year_month'].unique())

LGB_PARAMS = {
    'objective': 'tweedie', 'tweedie_variance_power': 1.5,
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
    'n_estimators': 300, 'verbose': -1, 'n_jobs': -1, 'random_state': 42
}

def wape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.maximum(np.array(y_pred), 0)
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-9) * 100

# We will test forecasting 12 months ahead from 3 different "Origin" months
cv_results = []
origin_offsets = [13, 14, 15] # Backtest from 13, 14, and 15 months ago

for fold, offset in enumerate(origin_offsets, 1):
    origin_month = all_months[-offset]
    print(f"\nFold {fold} | Forecast Origin: {origin_month.strftime('%Y-%m')}")
    
    fold_wapes = []
    # Train 12 separate models for this fold
    for h in HORIZONS:
        target_col = f'target_h{h}'
        
        # Train on data up to the origin (excluding rows where target is NaN)
        train_data = train_panel[(train_panel['year_month'] <= origin_month) & train_panel[target_col].notna()]
        X_tr, y_tr = train_data[FEATURE_COLS], train_data[target_col]
        
        model_h = lgb.LGBMRegressor(**LGB_PARAMS)
        model_h.fit(X_tr, y_tr)
        
        # Evaluate: Predict horizon h using features at the origin_month
        test_data = train_panel[train_panel['year_month'] == origin_month].dropna(subset=[target_col])
        if test_data.empty: continue
            
        X_te, y_te = test_data[FEATURE_COLS], test_data[target_col]
        preds = np.maximum(model_h.predict(X_te), 0)
        
        h_wape = wape(y_te, preds)
        fold_wapes.append(h_wape)
        
    avg_fold_wape = np.mean(fold_wapes)
    print(f"  → Fold Mean WAPE across 12 horizons: {avg_fold_wape:.2f}%")
    cv_results.append(avg_fold_wape)

print(f"\nOverall CV WAPE: {np.mean(cv_results):.2f}%")

# ─────────────────────────────────────────
# 5. TRAIN FINAL PRODUCTION MODELS (1 to 12)
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 4: Training 12 Final Models on Full History")
print("="*60)

final_models = {}

for h in HORIZONS:
    target_col = f'target_h{h}'
    train_data = train_panel.dropna(subset=[target_col])
    
    X_full = train_data[FEATURE_COLS]
    y_full = train_data[target_col]
    
    m = lgb.LGBMRegressor(**LGB_PARAMS)
    m.fit(X_full, y_full)
    final_models[h] = m
    print(f"  ✓ Model h={h} trained ({len(train_data)} rows)")

# ─────────────────────────────────────────
# 6. INFERENCE: ONE-SHOT PREDICTION
# ─────────────────────────────────────────
print("\n" + "="*60)
print("STEP 5: Generating 12-Month Forecast")
print("="*60)

# In a Direct Multi-Step architecture, inference is beautiful. 
# We simply take the features from the VERY LAST MONTH in our dataset, 
# and pass that exact same row of features into all 12 models.
LAST_DATE = panel['year_month'].max()
origin_features = panel[panel['year_month'] == LAST_DATE].copy()

forecast_rows = []

for h in HORIZONS:
    fdate = LAST_DATE + pd.DateOffset(months=h)
    
    # Use Model h to predict sales at LAST_DATE + h
    X_infer = origin_features[FEATURE_COLS]
    preds = np.maximum(final_models[h].predict(X_infer), 0)
    
    for i, (_, row) in enumerate(origin_features.iterrows()):
        forecast_rows.append({
            'model':          encoders['model'].inverse_transform([int(row['model'])])[0],
            'variant':        encoders['variant'].inverse_transform([int(row['variant'])])[0],
            'forecast_month': fdate.strftime('%Y-%m'),
            'horizon_h':      h,
            'forecast_units': round(preds[i], 2),
        })

forecast_df = pd.DataFrame(forecast_rows)
variant_fc = forecast_df.pivot_table(index=['model', 'variant'], columns='forecast_month', values='forecast_units').reset_index()
variant_fc.to_csv(OUT_DIR / "variant_forecast_direct12.csv", index=False)

print(f"\n✓ Output saved to {OUT_DIR}/variant_forecast_direct12.csv")
print("PIPELINE COMPLETE.")

STEP 1: Loading, aggregating, and densifying data
Densified panel shape: (94554, 17)

STEP 2: Feature Engineering & Target Creation
Total features per model: 39

STEP 3: Direct Multi-Step CV Evaluation

Fold 1 | Forecast Origin: 2024-10
  → Fold Mean WAPE across 12 horizons: 83.01%

Fold 2 | Forecast Origin: 2024-09
  → Fold Mean WAPE across 12 horizons: 85.90%

Fold 3 | Forecast Origin: 2024-08
  → Fold Mean WAPE across 12 horizons: 87.08%

Overall CV WAPE: 85.33%

STEP 4: Training 12 Final Models on Full History
  ✓ Model h=1 trained (21114 rows)
  ✓ Model h=2 trained (20808 rows)
  ✓ Model h=3 trained (20502 rows)
  ✓ Model h=4 trained (20196 rows)
  ✓ Model h=5 trained (19890 rows)
  ✓ Model h=6 trained (19584 rows)
  ✓ Model h=7 trained (19278 rows)
  ✓ Model h=8 trained (18972 rows)
  ✓ Model h=9 trained (18666 rows)
  ✓ Model h=10 trained (18360 rows)
  ✓ Model h=11 trained (18054 rows)
  ✓ Model h=12 trained (17748 rows)

STEP 5: Generating 12-Month Forecast

✓ Output saved to 